### This notebook provides an example to construct the feature matrix for phenotype cholesterol uptake with 6 target instances.
### Customize the cell type, phenotype, target list, positive instance list, and file paths as needed.
### Note: features from this notebook do not include diffusion scores. Diffusion scores would be added in baseline_scores.ipynb

In [28]:
import json
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import networkx as nx
import torch
from torch_geometric.utils.convert import from_networkx
import numpy as np
import merge_feature as mf
from stringdb_alias import HGNCMapper

In [29]:
mapper = HGNCMapper('../Data/Graphs/9606.protein.info.v11.5.txt.gz', '../Data/Graphs/9606.protein.aliases.v11.5.txt.gz')

In [30]:
import pickle
with open('../Data/Graphs/Physical_graph_HepG2.gpickle', 'rb') as f:
    ppi_graph = pickle.load(f)
ppi_graph.number_of_nodes()

17196

In [31]:
node_feature = mf.merge_all('../Data/Raw/LDLR/target_6/','node_features','_LDLR_target_6.json',0,1021)
## Convert dictionary to dataframe
feature_df = mf.dict2df(node_feature)

../Data/Raw/LDLR/target_6/node_features0_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features1_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features2_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features3_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features4_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features5_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features6_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features7_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features8_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features9_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features10_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features11_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features12_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features13_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features14_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_features15_LDLR_target_6.json
../Data/Raw/LDLR/target_6/node_fea

In [32]:
feature_df

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,DTDP,DDEP,DDTP,DDDP,has_path,num_paths,shortest_path_len,max_path_score,max_degree_score,protein
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,75,4,0.100635,1.540927e-06,9606.ENSP00000391488
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,64,4,0.162778,2.940264e-07,9606.ENSP00000361748
2,0,0,0,0,14,4,0,0,1,0,...,0,137,0,0,1,12688,3,0.876653,2.741228e-04,9606.ENSP00000356641
3,0,0,0,0,2,2,0,0,0,0,...,0,0,0,0,1,469,3,0.369317,8.549932e-05,9606.ENSP00000417581
4,0,0,0,0,0,0,0,0,3,0,...,3,14,2,3,1,898,3,0.737119,6.452445e-05,9606.ENSP00000365006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18379,0,0,0,0,25,9,1,0,0,0,...,0,0,0,0,1,20763,3,0.296059,6.199244e-05,9606.ENSP00000340610
18380,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,10000,0.000000,0.000000e+00,9606.ENSP00000370867
18381,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,738,4,0.313554,2.819650e-06,9606.ENSP00000354223
18382,0,1,0,0,11,7,1,4,0,0,...,0,0,0,0,1,9277,2,0.425124,3.300330e-03,9606.ENSP00000452401


In [33]:
feature_df['has_path'].value_counts()

has_path
1    17161
0     1223
Name: count, dtype: int64

In [34]:
feature_df['shortest_path_len'].value_counts()

shortest_path_len
3        13883
4         2507
10000     1223
2          730
5           35
1            6
Name: count, dtype: int64

In [35]:
# thermo
def thermometer_encoding(df,column_name):
    max_value = df[df[column_name]!=10000][column_name].max()
    # print(max_value)
    # df.loc[df[column_name]==10000,column_name] = max_value+1
    encoded = np.array([
        [1 if i < value else 0 for i in range(max_value)]
        for value in df[column_name]
    ])
    # Set the encoding for the maximum value as all 0's
    encoded[df[column_name] == 10000] = 0
    # print(encoded)
    encoded_df = pd.DataFrame(encoded, columns=[f"thermo{i+1}" for i in range(max_value)], index=df[column_name].index)
    df = pd.concat([df,encoded_df],axis = 1)
    return df


In [36]:
feature_df = thermometer_encoding(feature_df,'shortest_path_len')
feature_df

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,num_paths,shortest_path_len,max_path_score,max_degree_score,protein,thermo1,thermo2,thermo3,thermo4,thermo5
0,0,0,0,0,0,0,0,0,0,0,...,75,4,0.100635,1.540927e-06,9606.ENSP00000391488,1,1,1,1,0
1,0,0,0,0,0,0,0,0,0,0,...,64,4,0.162778,2.940264e-07,9606.ENSP00000361748,1,1,1,1,0
2,0,0,0,0,14,4,0,0,1,0,...,12688,3,0.876653,2.741228e-04,9606.ENSP00000356641,1,1,1,0,0
3,0,0,0,0,2,2,0,0,0,0,...,469,3,0.369317,8.549932e-05,9606.ENSP00000417581,1,1,1,0,0
4,0,0,0,0,0,0,0,0,3,0,...,898,3,0.737119,6.452445e-05,9606.ENSP00000365006,1,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18379,0,0,0,0,25,9,1,0,0,0,...,20763,3,0.296059,6.199244e-05,9606.ENSP00000340610,1,1,1,0,0
18380,0,0,0,0,0,0,0,0,0,0,...,0,10000,0.000000,0.000000e+00,9606.ENSP00000370867,0,0,0,0,0
18381,0,0,0,0,0,0,0,0,0,0,...,738,4,0.313554,2.819650e-06,9606.ENSP00000354223,1,1,1,1,0
18382,0,1,0,0,11,7,1,4,0,0,...,9277,2,0.425124,3.300330e-03,9606.ENSP00000452401,1,1,0,0,0


In [37]:
# set proteinID column the first column of this dataframe
protein_col = feature_df['protein']
feature_df.drop(labels=['protein'], axis=1, inplace=True)
feature_df.insert(0, 'protein', protein_col)

In [38]:
feature_df

,protein,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,...,has_path,num_paths,shortest_path_len,max_path_score,max_degree_score,thermo1,thermo2,thermo3,thermo4,thermo5
0,9606.ENSP00000391488,0,0,0,0,0,0,0,0,0,...,1,75,4,0.100635,1.540927e-06,1,1,1,1,0
1,9606.ENSP00000361748,0,0,0,0,0,0,0,0,0,...,1,64,4,0.162778,2.940264e-07,1,1,1,1,0
2,9606.ENSP00000356641,0,0,0,0,14,4,0,0,1,...,1,12688,3,0.876653,2.741228e-04,1,1,1,0,0
3,9606.ENSP00000417581,0,0,0,0,2,2,0,0,0,...,1,469,3,0.369317,8.549932e-05,1,1,1,0,0
4,9606.ENSP00000365006,0,0,0,0,0,0,0,0,3,...,1,898,3,0.737119,6.452445e-05,1,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18379,9606.ENSP00000340610,0,0,0,0,25,9,1,0,0,...,1,20763,3,0.296059,6.199244e-05,1,1,1,0,0
18380,9606.ENSP00000370867,0,0,0,0,0,0,0,0,0,...,0,0,10000,0.000000,0.000000e+00,0,0,0,0,0
18381,9606.ENSP00000354223,0,0,0,0,0,0,0,0,0,...,1,738,4,0.313554,2.819650e-06,1,1,1,1,0
18382,9606.ENSP00000452401,0,1,0,0,11,7,1,4,0,...,1,9277,2,0.425124,3.300330e-03,1,1,0,0,0


In [39]:
##Drop features that all nodes are the same
feature_df = mf.drop_same_features(feature_df)
feature_df

,protein,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,...,has_path,num_paths,shortest_path_len,max_path_score,max_degree_score,thermo1,thermo2,thermo3,thermo4,thermo5
0,9606.ENSP00000391488,0,0,0,0,0,0,0,0,0,...,1,75,4,0.100635,1.540927e-06,1,1,1,1,0
1,9606.ENSP00000361748,0,0,0,0,0,0,0,0,0,...,1,64,4,0.162778,2.940264e-07,1,1,1,1,0
2,9606.ENSP00000356641,0,0,0,0,14,4,0,0,1,...,1,12688,3,0.876653,2.741228e-04,1,1,1,0,0
3,9606.ENSP00000417581,0,0,0,0,2,2,0,0,0,...,1,469,3,0.369317,8.549932e-05,1,1,1,0,0
4,9606.ENSP00000365006,0,0,0,0,0,0,0,0,3,...,1,898,3,0.737119,6.452445e-05,1,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18379,9606.ENSP00000340610,0,0,0,0,25,9,1,0,0,...,1,20763,3,0.296059,6.199244e-05,1,1,1,0,0
18380,9606.ENSP00000370867,0,0,0,0,0,0,0,0,0,...,0,0,10000,0.000000,0.000000e+00,0,0,0,0,0
18381,9606.ENSP00000354223,0,0,0,0,0,0,0,0,0,...,1,738,4,0.313554,2.819650e-06,1,1,1,1,0
18382,9606.ENSP00000452401,0,1,0,0,11,7,1,4,0,...,1,9277,2,0.425124,3.300330e-03,1,1,0,0,0


In [40]:
feature_df['shortest_path_len'].value_counts()

shortest_path_len
3        13883
4         2507
10000     1223
2          730
5           35
1            6
Name: count, dtype: int64

In [41]:
# ##To thermometer encoding shortest path len, first try to find the longest shortest path len and make it 1 more longer
max_shortest_path_len = feature_df[feature_df['shortest_path_len']!=10000]['shortest_path_len'].max()
print(max_shortest_path_len)
feature_df.loc[feature_df['shortest_path_len'] == 10000, 'shortest_path_len'] =max_shortest_path_len+1
print(feature_df['shortest_path_len'].value_counts())
feature_df

5
shortest_path_len
3    13883
4     2507
6     1223
2      730
5       35
1        6
Name: count, dtype: int64


,protein,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,...,has_path,num_paths,shortest_path_len,max_path_score,max_degree_score,thermo1,thermo2,thermo3,thermo4,thermo5
0,9606.ENSP00000391488,0,0,0,0,0,0,0,0,0,...,1,75,4,0.100635,1.540927e-06,1,1,1,1,0
1,9606.ENSP00000361748,0,0,0,0,0,0,0,0,0,...,1,64,4,0.162778,2.940264e-07,1,1,1,1,0
2,9606.ENSP00000356641,0,0,0,0,14,4,0,0,1,...,1,12688,3,0.876653,2.741228e-04,1,1,1,0,0
3,9606.ENSP00000417581,0,0,0,0,2,2,0,0,0,...,1,469,3,0.369317,8.549932e-05,1,1,1,0,0
4,9606.ENSP00000365006,0,0,0,0,0,0,0,0,3,...,1,898,3,0.737119,6.452445e-05,1,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18379,9606.ENSP00000340610,0,0,0,0,25,9,1,0,0,...,1,20763,3,0.296059,6.199244e-05,1,1,1,0,0
18380,9606.ENSP00000370867,0,0,0,0,0,0,0,0,0,...,0,0,6,0.000000,0.000000e+00,0,0,0,0,0
18381,9606.ENSP00000354223,0,0,0,0,0,0,0,0,0,...,1,738,4,0.313554,2.819650e-06,1,1,1,1,0
18382,9606.ENSP00000452401,0,1,0,0,11,7,1,4,0,...,1,9277,2,0.425124,3.300330e-03,1,1,0,0,0


In [42]:
shortest_path_length = feature_df[['protein','shortest_path_len']]
shortest_path_length['shortest_path_len'] = -shortest_path_length['shortest_path_len']
shortest_path_length

/var/folders/0q/ws3979c565x6c64bbm5_vjz00000gn/T/ipykernel_31565/4242426199.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  shortest_path_length['shortest_path_len'] = -shortest_path_length['shortest_path_len']


,protein,shortest_path_len
0,9606.ENSP00000391488,-4
1,9606.ENSP00000361748,-4
2,9606.ENSP00000356641,-3
3,9606.ENSP00000417581,-3
4,9606.ENSP00000365006,-3
...,...,...
18379,9606.ENSP00000340610,-3
18380,9606.ENSP00000370867,-6
18381,9606.ENSP00000354223,-4
18382,9606.ENSP00000452401,-2


In [43]:
shortest_path_length['shortest_path_len'].value_counts()

shortest_path_len
-3    13883
-4     2507
-6     1223
-2      730
-5       35
-1        6
Name: count, dtype: int64

In [44]:
feature_df = mf.normalize_features(feature_df, ['protein'])
print(feature_df['has_path'].value_counts())
feature_df

has_path
1.0    17161
0.0     1223
Name: count, dtype: int64


,protein,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,...,has_path,num_paths,shortest_path_len,max_path_score,max_degree_score,thermo1,thermo2,thermo3,thermo4,thermo5
0,9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.0,0.000888,0.6,0.100635,1.540927e-06,1.0,1.0,1.0,1.0,0.0
1,9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.0,0.000757,0.6,0.162778,2.940264e-07,1.0,1.0,1.0,1.0,0.0
2,9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,...,1.0,0.150157,0.4,0.876653,2.741228e-04,1.0,1.0,1.0,0.0,0.0
3,9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,...,1.0,0.005550,0.4,0.369317,8.549932e-05,1.0,1.0,1.0,0.0,0.0
4,9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,...,1.0,0.010627,0.4,0.737119,6.452445e-05,1.0,1.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18379,9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,...,1.0,0.245722,0.4,0.296059,6.199244e-05,1.0,1.0,1.0,0.0,0.0
18380,9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,1.0,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.0
18381,9606.ENSP00000354223,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.0,0.008734,0.6,0.313554,2.819650e-06,1.0,1.0,1.0,1.0,0.0
18382,9606.ENSP00000452401,0.0,0.333333,0.0,0.0,0.064706,0.114754,0.055556,0.111111,0.000000,...,1.0,0.109790,0.2,0.425124,3.300330e-03,1.0,1.0,0.0,0.0,0.0


In [ ]:
positive = list(np.load('../Data/Raw/LDLR/LDLR_pos_gene.npy'))
len(positive)

473

In [46]:
train_pro = list(np.load('../Data/Raw/LDLR/LDLR_train_pro.npy'))
len(train_pro)

17768

In [47]:
# add label
feature_df = mf.add_label(feature_df,positive)
print(feature_df['label'].value_counts())
feature_df

label
0.0    17911
1.0      473
Name: count, dtype: int64


,protein,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,...,num_paths,shortest_path_len,max_path_score,max_degree_score,thermo1,thermo2,thermo3,thermo4,thermo5,label
0,9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000888,0.6,0.100635,1.540927e-06,1.0,1.0,1.0,1.0,0.0,0.0
1,9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000757,0.6,0.162778,2.940264e-07,1.0,1.0,1.0,1.0,0.0,0.0
2,9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,...,0.150157,0.4,0.876653,2.741228e-04,1.0,1.0,1.0,0.0,0.0,0.0
3,9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,...,0.005550,0.4,0.369317,8.549932e-05,1.0,1.0,1.0,0.0,0.0,0.0
4,9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,...,0.010627,0.4,0.737119,6.452445e-05,1.0,1.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18379,9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,...,0.245722,0.4,0.296059,6.199244e-05,1.0,1.0,1.0,0.0,0.0,0.0
18380,9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.0,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0
18381,9606.ENSP00000354223,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.008734,0.6,0.313554,2.819650e-06,1.0,1.0,1.0,1.0,0.0,0.0
18382,9606.ENSP00000452401,0.0,0.333333,0.0,0.0,0.064706,0.114754,0.055556,0.111111,0.000000,...,0.109790,0.2,0.425124,3.300330e-03,1.0,1.0,0.0,0.0,0.0,0.0


In [48]:
shortest_path_length = mf.add_label(shortest_path_length,positive)
shortest_path_length = mf.add_mask(shortest_path_length,train_pro)
print(shortest_path_length['label'].value_counts())
shortest_path_length = shortest_path_length.set_index('protein')
shortest_path_length = shortest_path_length[shortest_path_length['train_mask']==True]
shortest_path_length = shortest_path_length.drop(['train_mask','val_mask','test_mask'],axis = 1)
print(shortest_path_length['label'].value_counts())
# shortest_path_length.to_csv('/Users/leojin/Desktop/IGVF/Data/Datasets/LDLR/shortest_path_length_data_target6.csv')

/Users/leojin/Desktop/IGVF/Code/merge_feature.py:179: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  feature_df.loc[i, 'label'] = 0
/Users/leojin/Desktop/IGVF/Code/merge_feature.py:185: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  feature_df.loc[i, 'train_mask'] = True
/Users/leojin/Desktop/IGVF/Code/merge_feature.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: h

label
0.0    17911
1.0      473
Name: count, dtype: int64
label
0.0    17295
1.0      473
Name: count, dtype: int64


In [49]:
feature_df = mf.add_mask(feature_df,train_pro)
print(feature_df['train_mask'].value_counts())
feature_df

train_mask
True     17768
False      616
Name: count, dtype: int64


,protein,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,...,max_degree_score,thermo1,thermo2,thermo3,thermo4,thermo5,label,train_mask,val_mask,test_mask
0,9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.540927e-06,1.0,1.0,1.0,1.0,0.0,0.0,True,False,False
1,9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,2.940264e-07,1.0,1.0,1.0,1.0,0.0,0.0,True,False,False
2,9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,...,2.741228e-04,1.0,1.0,1.0,0.0,0.0,0.0,True,False,False
3,9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,...,8.549932e-05,1.0,1.0,1.0,0.0,0.0,0.0,True,False,False
4,9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,...,6.452445e-05,1.0,1.0,1.0,0.0,0.0,0.0,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18379,9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,...,6.199244e-05,1.0,1.0,1.0,0.0,0.0,0.0,True,False,False
18380,9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,True,False,False
18381,9606.ENSP00000354223,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,2.819650e-06,1.0,1.0,1.0,1.0,0.0,0.0,True,False,False
18382,9606.ENSP00000452401,0.0,0.333333,0.0,0.0,0.064706,0.114754,0.055556,0.111111,0.000000,...,3.300330e-03,1.0,1.0,0.0,0.0,0.0,0.0,False,False,False


In [50]:
feature_df = mf.move2last(feature_df,'label')
feature_df = feature_df.set_index('protein')
feature_df

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,max_degree_score,thermo1,thermo2,thermo3,thermo4,thermo5,train_mask,val_mask,test_mask,label
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,1.540927e-06,1.0,1.0,1.0,1.0,0.0,True,False,False,0.0
9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,2.940264e-07,1.0,1.0,1.0,1.0,0.0,True,False,False,0.0
9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,0.0,...,2.741228e-04,1.0,1.0,1.0,0.0,0.0,True,False,False,0.0
9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,0.0,...,8.549932e-05,1.0,1.0,1.0,0.0,0.0,True,False,False,0.0
9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,0.0,...,6.452445e-05,1.0,1.0,1.0,0.0,0.0,True,False,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,0.0,...,6.199244e-05,1.0,1.0,1.0,0.0,0.0,True,False,False,0.0
9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000e+00,0.0,0.0,0.0,0.0,0.0,True,False,False,0.0
9606.ENSP00000354223,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,2.819650e-06,1.0,1.0,1.0,1.0,0.0,True,False,False,0.0


In [51]:
feature_df = feature_df[feature_df['train_mask']==True]
feature_df

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,max_degree_score,thermo1,thermo2,thermo3,thermo4,thermo5,train_mask,val_mask,test_mask,label
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,1.540927e-06,1.0,1.0,1.0,1.0,0.0,True,False,False,0.0
9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,2.940264e-07,1.0,1.0,1.0,1.0,0.0,True,False,False,0.0
9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,0.0,...,2.741228e-04,1.0,1.0,1.0,0.0,0.0,True,False,False,0.0
9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,0.0,...,8.549932e-05,1.0,1.0,1.0,0.0,0.0,True,False,False,0.0
9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,0.0,...,6.452445e-05,1.0,1.0,1.0,0.0,0.0,True,False,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9606.ENSP00000356694,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000e+00,0.0,0.0,0.0,0.0,0.0,True,False,False,0.0
9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,0.0,...,6.199244e-05,1.0,1.0,1.0,0.0,0.0,True,False,False,0.0
9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000e+00,0.0,0.0,0.0,0.0,0.0,True,False,False,0.0


In [52]:
feature_df = feature_df.drop(['train_mask','val_mask','test_mask'],axis=1)
feature_df

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,num_paths,shortest_path_len,max_path_score,max_degree_score,thermo1,thermo2,thermo3,thermo4,thermo5,label
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000888,0.6,0.100635,1.540927e-06,1.0,1.0,1.0,1.0,0.0,0.0
9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000757,0.6,0.162778,2.940264e-07,1.0,1.0,1.0,1.0,0.0,0.0
9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,0.0,...,0.150157,0.4,0.876653,2.741228e-04,1.0,1.0,1.0,0.0,0.0,0.0
9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,0.0,...,0.005550,0.4,0.369317,8.549932e-05,1.0,1.0,1.0,0.0,0.0,0.0
9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,0.0,...,0.010627,0.4,0.737119,6.452445e-05,1.0,1.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9606.ENSP00000356694,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,1.0,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0
9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,0.0,...,0.245722,0.4,0.296059,6.199244e-05,1.0,1.0,1.0,0.0,0.0,0.0
9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,1.0,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0


In [53]:
# combine extracted property and abundance features
abundance_feature = pd.read_csv('../Data/Raw/cellular-localization/gene_abundances_HepG2.csv', index_col=0)[['RNA line ab','RNA type ab','binary_RNA line ab','binary_RNA type ab','one_hot_0','one_hot_1','one_hot_2','one_hot_3']]
## For other cell types, you may only use "RNA line ab" and  "binary RNA line ab"
abundance_feature_first = abundance_feature.loc[
    ~abundance_feature.index.duplicated(keep='first')
]
# abundance_feature = pd.read_csv('/Users/leojin/Desktop/IGVF/Data/Raw/cellular-localization/gene_abundances_HepG2.csv', index_col=0)
feature_df = pd.merge(feature_df,abundance_feature_first,left_index = True,right_index = True,how = 'left')
feature_df = mf.move2last(feature_df,'label')
feature_df.fillna(0,inplace=True)
feature_df
# feature_df = feature_df.set_index('protein')
# feature_df.to_csv('/Users/leojin/Desktop/IGVF/Influenza/train_node_target1.csv')

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,thermo5,RNA line ab,RNA type ab,binary_RNA line ab,binary_RNA type ab,one_hot_0,one_hot_1,one_hot_2,one_hot_3,label
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.6,3.0,1.0,1.0,0.25,0.25,0.25,0.25,0.0
9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,44.2,51.8,1.0,1.0,0.00,0.00,1.00,0.00,0.0
9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0
9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,0.0,...,0.0,12.5,15.7,1.0,1.0,1.00,0.00,0.00,0.00,0.0
9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,0.0,...,0.0,0.1,1.8,1.0,1.0,0.25,0.25,0.25,0.25,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9606.ENSP00000356694,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,1.00,0.00,0.00,0.00,0.0
9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.25,0.25,0.0
9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,1.00,0.00,0.00,0.00,0.0


### Add subcellular localization features

In [54]:
def get_voc(df,col_to_add,split_sign=':'):
    df[['first','second','third']] = df[col_to_add].str.split(split_sign,expand=True)
    voc = list()
    voc.append(df['first'].unique().tolist())
    voc.append(df['second'].unique().tolist())
    voc.append(df['third'].unique().tolist())
    voc = list(set([item for sublist in voc for item in sublist]))
    if '' in voc:
        voc.remove('')
    voc = [item for item in voc if item is not None]
    return voc

def add_subcell(feature_df,subcell_df,col_to_add,map_col):
    voc = get_voc(sub_cellular_dataset,col_to_add)
    # map_index = sub_cellular_dataset.columns.get_loc(map_col)
    # print(map_index)
    feature_df.loc[:,voc] = 0
    feature_df.loc[:,'miss_subcell'] = 1
    for i in range(len(subcell_df)):
        string = subcell_df.iloc[i].loc[map_col]
        if string in feature_df.index:
            feature_df.loc[string,'miss_subcell'] = 0
            locations = subcell_df.iloc[i,].loc[col_to_add].split(':')
            if '' in locations:
                locations.remove('')
            for j in locations:
                feature_df.loc[string,j] = 1
    feature_df = mf.move2last(feature_df,'label')
    return feature_df
        
    
    

In [55]:
sub_cellular_dataset = pd.read_csv('/Users/leojin/Desktop/IGVF/Data/Raw/uniprot_reactome_hpa_merged_stringid.csv',index_col = 0)
subcell_feature_group = get_voc(sub_cellular_dataset,'hpa_location')
subcell_df = pd.DataFrame(index = sub_cellular_dataset['StringID'].unique().tolist(),columns = subcell_feature_group)
subcell_df

,cytokinetic bridge,intermediate filaments,cytoplasm,actin filaments,centrosome,nuclear speckles,secreted proteins,cleavage furrow,peroxisomes,kinetochore,...,plasma membrane,mitotic spindle,nuclear bodies,aggresome,nucleoli rim,microtubule ends,focal adhesion sites,rods & rings,lipid droplets,nucleoplasm
9606.ENSP00000484893,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000377112,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000371212,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000419279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000372394,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9606.ENSP00000485668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000400713,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000290390,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000429022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [56]:
feature_df = add_subcell(feature_df,sub_cellular_dataset,'hpa_location','StringID')
feature_df
# feature_df.to_csv('/Users/leojin/Desktop/IGVF/Data/Datasets/LDLR/train_node_target6.csv')

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,nuclear bodies,aggresome,nucleoli rim,microtubule ends,focal adhesion sites,rods & rings,lipid droplets,nucleoplasm,miss_subcell,label
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0,0,0,0,0,0,0,0,0,0.0
9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0,0,0,0,0,0,0,1,0,0.0
9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,0.0,...,0,0,0,0,0,0,0,0,1,0.0
9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,0.0,...,0,0,0,0,0,0,0,1,0,0.0
9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,0.0,...,0,0,0,0,0,0,0,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9606.ENSP00000356694,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0,0,0,0,0,0,0,0,0,0.0
9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,0.0,...,0,0,0,0,0,0,0,0,1,0.0
9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0,0,0,0,0,0,0,0,0,0.0


### Add GO Embeddings

In [57]:
go_embedding = pd.read_csv('/Users/leojin/Desktop/IGVF/Data/Raw/go_embedding_64.csv',index_col= 'string_id')
go_embedding = go_embedding.drop(['Unnamed: 0'],axis = 1)
merged_df = pd.merge(feature_df, go_embedding, left_index=True, right_index=True, how='left')
missing_values = merged_df.isna().any(axis=1)
merged_df.fillna(0,inplace = True)
# Convert boolean values to 1 for missing values and 0 for non-missing values
merged_df['Has_Missing'] = missing_values.astype(int)
merged_df = mf.move2last(merged_df,'label')
merged_df
# 

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,go_feature_56,go_feature_57,go_feature_58,go_feature_59,go_feature_60,go_feature_61,go_feature_62,go_feature_63,Has_Missing,label
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.073982,0.463450,0.228938,0.304010,-0.014238,-0.626229,-0.066336,1.176438,0,0.0
9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,-0.204774,0.409705,-0.187285,-0.258663,-0.194244,0.116084,0.076689,0.474523,0,0.0
9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,0.0,...,0.011795,-0.107706,-0.194522,-0.004723,-0.106734,-0.069825,0.016390,-0.146723,0,0.0
9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,0.0,...,-0.242968,0.114746,0.077302,0.250145,0.056601,-0.577621,-0.042117,1.328747,0,0.0
9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,0.0,...,-0.567111,0.109618,0.318059,-0.036521,-0.123925,-0.405910,-0.206974,-0.314821,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9606.ENSP00000356694,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,-0.146530,-0.793537,-0.464282,0.003613,-0.165405,0.090635,0.435142,0.156999,0,0.0
9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1,0.0
9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.300933,0.075464,-0.070108,-0.229924,-0.125044,-0.100176,0.009644,-0.103494,0,0.0


### Add similarity as additional feature (need to hardcode the index of subcellular localization and go-features)

In [58]:
def cosine_similarity(vec_a,vec_b):
    dot_product = np.dot(vec_a,vec_b)
    magnitude_a = np.linalg.norm(vec_a)
    magnitude_b = np.linalg.norm(vec_b)
    similarity = dot_product / (magnitude_a * magnitude_b)
    return similarity

In [59]:
target = list(np.load('../Data/Raw/targets/LDLR_target_6.npy'))
target_feature = merged_df.loc[target]
target_feature

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,go_feature_56,go_feature_57,go_feature_58,go_feature_59,go_feature_60,go_feature_61,go_feature_62,go_feature_63,Has_Missing,label
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000454071,1.0,0.666667,0.333333,0.0,0.500000,0.245902,0.111111,0.250000,0.421053,0.166667,...,-0.152231,-0.077891,-0.189463,0.081509,0.028242,0.177071,0.055454,0.126733,0,1.0
9606.ENSP00000363458,1.0,0.666667,0.333333,0.0,0.194118,0.377049,0.222222,0.055556,0.000000,0.000000,...,-0.157977,0.166201,0.295280,0.033997,0.006775,-0.021339,0.086764,-0.224559,0,0.0
9606.ENSP00000261349,1.0,0.333333,0.666667,0.0,0.470588,0.262295,0.055556,0.388889,0.315789,0.333333,...,0.128736,0.046213,0.133114,-0.008198,-0.077056,0.138898,-0.141427,0.056520,0,0.0
9606.ENSP00000344155,1.0,0.000000,0.000000,0.0,0.017647,0.000000,0.000000,0.055556,0.000000,0.166667,...,-0.344231,0.184484,0.102964,-0.124796,-0.069071,-0.191371,-0.186445,-0.075528,0,0.0
9606.ENSP00000346032,1.0,0.333333,0.333333,0.0,0.105882,0.049180,0.000000,0.444444,0.315789,0.000000,...,-0.039972,-0.404266,0.101614,-0.325349,0.315884,-0.049999,-0.231966,0.120836,0,0.0
9606.ENSP00000303208,1.0,0.666667,0.333333,0.0,0.041176,0.081967,0.222222,0.527778,0.263158,0.166667,...,-0.152454,0.380297,-0.367007,-0.449202,-0.048670,0.426443,0.045150,-0.039103,0,0.0


In [60]:
target_go = target_feature.iloc[:,99:164]
target_go

,go_feature_0,go_feature_1,go_feature_2,go_feature_3,go_feature_4,go_feature_5,go_feature_6,go_feature_7,go_feature_8,go_feature_9,...,go_feature_55,go_feature_56,go_feature_57,go_feature_58,go_feature_59,go_feature_60,go_feature_61,go_feature_62,go_feature_63,Has_Missing
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000454071,-0.203689,-0.061754,-0.090591,0.217105,-0.075774,0.108297,-0.122949,0.049393,-0.012973,-0.044842,...,-0.313521,-0.152231,-0.077891,-0.189463,0.081509,0.028242,0.177071,0.055454,0.126733,0
9606.ENSP00000363458,-0.077594,0.396221,0.338874,0.325630,0.132866,0.117838,-0.387007,0.549298,-0.203616,-0.798561,...,-0.035291,-0.157977,0.166201,0.295280,0.033997,0.006775,-0.021339,0.086764,-0.224559,0
9606.ENSP00000261349,0.019141,-0.205365,-0.128900,-0.187337,-0.063899,0.355608,-0.250694,0.013855,0.016402,0.032574,...,0.140564,0.128736,0.046213,0.133114,-0.008198,-0.077056,0.138898,-0.141427,0.056520,0
9606.ENSP00000344155,-0.203278,-0.130895,0.132085,0.224173,0.034432,-0.051353,0.065071,-0.169145,0.154375,0.115930,...,0.021629,-0.344231,0.184484,0.102964,-0.124796,-0.069071,-0.191371,-0.186445,-0.075528,0
9606.ENSP00000346032,-0.123994,-0.321591,-0.398896,-0.109315,-0.080759,0.050866,0.215804,0.771391,0.022277,-0.317005,...,0.476658,-0.039972,-0.404266,0.101614,-0.325349,0.315884,-0.049999,-0.231966,0.120836,0
9606.ENSP00000303208,-0.037354,-0.446068,0.509243,-0.306779,0.167861,-0.144595,0.304649,0.048073,0.254676,-0.428546,...,-0.226273,-0.152454,0.380297,-0.367007,-0.449202,-0.048670,0.426443,0.045150,-0.039103,0


In [61]:

target_sub = target_feature.iloc[:,58:99]
target_sub


,cytokinetic bridge,intermediate filaments,cytoplasm,actin filaments,centrosome,nuclear speckles,secreted proteins,cleavage furrow,peroxisomes,kinetochore,...,mitotic spindle,nuclear bodies,aggresome,nucleoli rim,microtubule ends,focal adhesion sites,rods & rings,lipid droplets,nucleoplasm,miss_subcell
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000454071,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9606.ENSP00000363458,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9606.ENSP00000261349,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9606.ENSP00000344155,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9606.ENSP00000346032,0,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9606.ENSP00000303208,0,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [62]:
node_feature = merged_df.copy()

In [63]:
node_feature['go_sim'] = 0
node_feature['sub_sim'] = 0

In [64]:
node_feature

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,go_feature_58,go_feature_59,go_feature_60,go_feature_61,go_feature_62,go_feature_63,Has_Missing,label,go_sim,sub_sim
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.228938,0.304010,-0.014238,-0.626229,-0.066336,1.176438,0,0.0,0,0
9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,-0.187285,-0.258663,-0.194244,0.116084,0.076689,0.474523,0,0.0,0,0
9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,0.0,...,-0.194522,-0.004723,-0.106734,-0.069825,0.016390,-0.146723,0,0.0,0,0
9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,0.0,...,0.077302,0.250145,0.056601,-0.577621,-0.042117,1.328747,0,0.0,0,0
9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,0.0,...,0.318059,-0.036521,-0.123925,-0.405910,-0.206974,-0.314821,0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9606.ENSP00000356694,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,-0.464282,0.003613,-0.165405,0.090635,0.435142,0.156999,0,0.0,0,0
9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1,0.0,0,0
9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,-0.070108,-0.229924,-0.125044,-0.100176,0.009644,-0.103494,0,0.0,0,0


In [65]:
for i in range(len(node_feature)):
    instance = node_feature.iloc[i].name
    source_vec_go = node_feature.iloc[i,99:164]
    source_vec_sub = node_feature.iloc[i,58:99]
    # print(max(target_go.dot(source_vec_go) / (np.linalg.norm(target_go, axis=1) * np.linalg.norm(source_vec_go))),max(target_sub.dot(source_vec_sub) / (np.linalg.norm(target_sub, axis=1) * np.linalg.norm(source_vec_sub))))
    node_feature.loc[instance,'go_sim'] = max(target_go.dot(source_vec_go) / (np.linalg.norm(target_go, axis=1) * np.linalg.norm(source_vec_go)))
    node_feature.loc[instance,'sub_sim'] = max(target_sub.dot(source_vec_sub) / (np.linalg.norm(target_sub, axis=1) * np.linalg.norm(source_vec_sub)))


/var/folders/0q/ws3979c565x6c64bbm5_vjz00000gn/T/ipykernel_31565/2641054422.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.0487702072843169' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  node_feature.loc[instance,'go_sim'] = max(target_go.dot(source_vec_go) / (np.linalg.norm(target_go, axis=1) * np.linalg.norm(source_vec_go)))
/var/folders/0q/ws3979c565x6c64bbm5_vjz00000gn/T/ipykernel_31565/2641054422.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.31622776601683794' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  node_feature.loc[instance,'sub_sim'] = max(target_sub.dot(source_vec_sub) / (np.linalg.norm(target_sub, axis=1) * np.linalg.norm(source_vec_sub)))


In [66]:
node_feature

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,go_feature_58,go_feature_59,go_feature_60,go_feature_61,go_feature_62,go_feature_63,Has_Missing,label,go_sim,sub_sim
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.228938,0.304010,-0.014238,-0.626229,-0.066336,1.176438,0,0.0,0.048770,0.316228
9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,-0.187285,-0.258663,-0.194244,0.116084,0.076689,0.474523,0,0.0,0.043036,0.559017
9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,0.0,...,-0.194522,-0.004723,-0.106734,-0.069825,0.016390,-0.146723,0,0.0,-0.026264,0.000000
9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,0.0,...,0.077302,0.250145,0.056601,-0.577621,-0.042117,1.328747,0,0.0,0.069183,0.500000
9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,0.0,...,0.318059,-0.036521,-0.123925,-0.405910,-0.206974,-0.314821,0,0.0,0.408812,0.632456
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9606.ENSP00000356694,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,-0.464282,0.003613,-0.165405,0.090635,0.435142,0.156999,0,0.0,0.136942,0.730297
9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1,0.0,0.000000,0.000000
9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,-0.070108,-0.229924,-0.125044,-0.100176,0.009644,-0.103494,0,0.0,0.174566,0.447214


In [67]:
node_feature = mf.move2last(node_feature,'label')

In [68]:
target_feature = node_feature.loc[target]
target_feature['go_sim'] = target_feature['go_sim']

In [69]:
target_feature['go_sim'].value_counts()

go_sim
1.0    2
1.0    2
1.0    1
1.0    1
Name: count, dtype: int64

## Concatenate Target feature vectors

In [70]:
phenotype = 'LDLR'
targets = list(np.load('/Users/leojin/Desktop/IGVF/Data/Raw/targets/'+phenotype+'_target_6.npy'))

In [71]:
def get_voc(df,col_to_add,split_sign=':'):
    df[['first','second','third']] = df[col_to_add].str.split(split_sign,expand=True)
    voc = list()
    voc.append(df['first'].unique().tolist())
    voc.append(df['second'].unique().tolist())
    voc.append(df['third'].unique().tolist())
    voc = list(set([item for sublist in voc for item in sublist]))
    if '' in voc:
        voc.remove('')
    voc = [item for item in voc if item is not None]
    return voc

def add_subcell(feature_df,subcell_df,col_to_add,map_col,voc):
    voc = get_voc(subcell_df,col_to_add)
    # map_index = sub_cellular_dataset.columns.get_loc(map_col)
    # print(map_index)
    feature_df.loc[:,voc] = 0
    feature_df.loc[:,'miss_subcell'] = 1
    for i in range(len(subcell_df)):
        string = subcell_df.iloc[i].loc[map_col]
        if string in feature_df.index:
            feature_df.loc[string,'miss_subcell'] = 0
            locations = subcell_df.iloc[i,].loc[col_to_add].split(':')
            if '' in locations:
                locations.remove('')
            for j in locations:
                feature_df.loc[string,j] = 1
    feature_df = mf.move2last(feature_df,'label')
    return feature_df
        
    
    

In [72]:
sub_cellular_dataset = pd.read_csv('/Users/leojin/Desktop/IGVF/Data/Raw/uniprot_reactome_hpa_merged_stringid.csv',index_col = 0)
subcell_feature_group = get_voc(sub_cellular_dataset,'hpa_location')
subcell_df = pd.DataFrame(index = sub_cellular_dataset['StringID'].unique().tolist(),columns = subcell_feature_group)
subcell_df

,cytokinetic bridge,intermediate filaments,cytoplasm,actin filaments,centrosome,nuclear speckles,secreted proteins,cleavage furrow,peroxisomes,kinetochore,...,plasma membrane,mitotic spindle,nuclear bodies,aggresome,nucleoli rim,microtubule ends,focal adhesion sites,rods & rings,lipid droplets,nucleoplasm
9606.ENSP00000484893,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000377112,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000371212,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000419279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000372394,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9606.ENSP00000485668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000400713,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000290390,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9606.ENSP00000429022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [73]:
phenotype = 'LDLR'
targets = list(np.load('/Users/leojin/Desktop/IGVF/Data/Raw/targets/'+phenotype+'_target_6.npy'))
target_with_go = list(set(targets).intersection(set((pd.read_csv('/Users/leojin/Desktop/IGVF/Data/Raw/Go_embeddings/go_embedding_64_string.csv',index_col = 0).index))))
target_abundance = pd.read_csv('/Users/leojin/Desktop/IGVF/Data/Raw/cellular-localization/gene_abundances_HepG2.csv',index_col = 'protein').loc[targets,]
target_go_embedding = pd.read_csv('/Users/leojin/Desktop/IGVF/Data/Raw/Go_embeddings/go_embedding_64_string.csv',index_col = 0).loc[target_with_go,]
sub_cellular_dataset = pd.read_csv('/Users/leojin/Desktop/IGVF/Data/Raw/uniprot_reactome_hpa_merged_stringid.csv')
subcell_feature_group = get_voc(sub_cellular_dataset,'hpa_location')
subcell_df = pd.DataFrame(index = targets,columns = subcell_feature_group)
target_subcell = add_subcell(subcell_df,sub_cellular_dataset,'hpa_location','StringID','subcell_feature_group')
target_subcell = target_subcell.drop('label',axis=1)
target_df = pd.merge(target_subcell,target_abundance,left_index=True,right_index = True,how = 'inner')
target_df = pd.merge(target_df,target_go_embedding,left_index=True,right_index = True,how = 'inner')
target_df = pd.DataFrame(target_df.mean()).T
target_df.index = ['Target']
target_df[subcell_feature_group] = target_df[subcell_feature_group].where(target_df[subcell_feature_group] <= 0.0, 1)
target_df


,cytokinetic bridge,intermediate filaments,cytoplasm,actin filaments,centrosome,nuclear speckles,secreted proteins,cleavage furrow,peroxisomes,kinetochore,...,go_feature_54,go_feature_55,go_feature_56,go_feature_57,go_feature_58,go_feature_59,go_feature_60,go_feature_61,go_feature_62,go_feature_63
Target,0.0,0.0,1,0.0,0.0,0.0,1,0.0,0.0,0.0,...,-0.115946,0.010628,-0.119688,0.049173,0.01275,-0.132007,0.026017,0.079951,-0.062079,-0.00585


In [74]:
node_feature

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,go_feature_58,go_feature_59,go_feature_60,go_feature_61,go_feature_62,go_feature_63,Has_Missing,go_sim,sub_sim,label
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.228938,0.304010,-0.014238,-0.626229,-0.066336,1.176438,0,0.048770,0.316228,0.0
9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,-0.187285,-0.258663,-0.194244,0.116084,0.076689,0.474523,0,0.043036,0.559017,0.0
9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,0.0,...,-0.194522,-0.004723,-0.106734,-0.069825,0.016390,-0.146723,0,-0.026264,0.000000,0.0
9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,0.0,...,0.077302,0.250145,0.056601,-0.577621,-0.042117,1.328747,0,0.069183,0.500000,0.0
9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,0.0,...,0.318059,-0.036521,-0.123925,-0.405910,-0.206974,-0.314821,0,0.408812,0.632456,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9606.ENSP00000356694,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,-0.464282,0.003613,-0.165405,0.090635,0.435142,0.156999,0,0.136942,0.730297,0.0
9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1,0.000000,0.000000,0.0
9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,-0.070108,-0.229924,-0.125044,-0.100176,0.009644,-0.103494,0,0.174566,0.447214,0.0


In [75]:
a = pd.concat([target_df] * len(node_feature), ignore_index=True)
a.index = node_feature.index  # Set the index instead of reindexing
a = a.add_suffix('_y')
b = pd.concat([node_feature, a],axis = 1)
# b
node_attribute = mf.move2last(b,'label')
node_attribute
# node_attribute.to_csv('/Users/leojin/Desktop/IGVF/Data/Datasets/'+phenotype+'/train_node_attribute_'+phenotype+'_with_target_feature.csv')

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,go_feature_55_y,go_feature_56_y,go_feature_57_y,go_feature_58_y,go_feature_59_y,go_feature_60_y,go_feature_61_y,go_feature_62_y,go_feature_63_y,label
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.010628,-0.119688,0.049173,0.01275,-0.132007,0.026017,0.079951,-0.062079,-0.00585,0.0
9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.010628,-0.119688,0.049173,0.01275,-0.132007,0.026017,0.079951,-0.062079,-0.00585,0.0
9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,0.0,...,0.010628,-0.119688,0.049173,0.01275,-0.132007,0.026017,0.079951,-0.062079,-0.00585,0.0
9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,0.0,...,0.010628,-0.119688,0.049173,0.01275,-0.132007,0.026017,0.079951,-0.062079,-0.00585,0.0
9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,0.0,...,0.010628,-0.119688,0.049173,0.01275,-0.132007,0.026017,0.079951,-0.062079,-0.00585,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9606.ENSP00000356694,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.010628,-0.119688,0.049173,0.01275,-0.132007,0.026017,0.079951,-0.062079,-0.00585,0.0
9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,0.0,...,0.010628,-0.119688,0.049173,0.01275,-0.132007,0.026017,0.079951,-0.062079,-0.00585,0.0
9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.010628,-0.119688,0.049173,0.01275,-0.132007,0.026017,0.079951,-0.062079,-0.00585,0.0


### Add cellular abundance similarity scores to all datasets

In [76]:
def cosine_similarity(vec_a,vec_b):
    dot_product = np.dot(vec_a,vec_b)
    magnitude_a = np.linalg.norm(vec_a)
    magnitude_b = np.linalg.norm(vec_b)
    similarity = dot_product / (magnitude_a * magnitude_b)
    return similarity

In [ ]:
# phenotype = 'Influenza'
cell_type = 'HepG2'
cell_feature_name = ['RNA line ab', 'RNA type ab','binary_RNA line ab','binary_RNA type ab','one_hot_0','one_hot_1','one_hot_2','one_hot_3']
# cell_feature_name = ['RNA line ab','binary_RNA line ab']
target = np.load('/Users/leojin/Desktop/IGVF/Data/Raw/targets/'+phenotype+'_target_6.npy')
data = node_attribute.copy()
data['cell_ab_diff'] = data['RNA line ab'] - data['RNA line ab_y']
data['cell_ab_diff_abs'] = np.abs(data['cell_ab_diff'])
data['cell_type_ab_diff'] = data['RNA type ab'] - data['RNA type ab_y']
data['cell_type_ab_diff_abs'] = np.abs(data['cell_type_ab_diff'])
conditions = [
    data['one_hot_0'] == 1.0,
    data['one_hot_1'] == 1.0,
    data['one_hot_2'] == 1.0,
    data['one_hot_3'] == 1.0
]

choices = [2, 1, 0, 1]

data['cell_protein_ab_diff'] = np.select(conditions, choices, default=-1)
data = mf.move2last(data,'label')
data

,P,EP,TP,DP,EEP,ETP,EDP,TEP,TTP,TDP,...,go_feature_60_y,go_feature_61_y,go_feature_62_y,go_feature_63_y,cell_ab_diff,cell_ab_diff_abs,cell_type_ab_diff,cell_type_ab_diff_abs,cell_protein_ab_diff,label
protein,,,,,,,,,,,,,,,,,,,,,
9606.ENSP00000391488,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.026017,0.079951,-0.062079,-0.00585,-165.3,165.3,-44.916667,44.916667,-1,0.0
9606.ENSP00000361748,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.026017,0.079951,-0.062079,-0.00585,-121.7,121.7,3.883333,3.883333,0,0.0
9606.ENSP00000356641,0.0,0.000000,0.0,0.0,0.082353,0.065574,0.000000,0.000000,0.026316,0.0,...,0.026017,0.079951,-0.062079,-0.00585,-165.9,165.9,-47.916667,47.916667,-1,0.0
9606.ENSP00000417581,0.0,0.000000,0.0,0.0,0.011765,0.032787,0.000000,0.000000,0.000000,0.0,...,0.026017,0.079951,-0.062079,-0.00585,-153.4,153.4,-32.216667,32.216667,2,0.0
9606.ENSP00000365006,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.078947,0.0,...,0.026017,0.079951,-0.062079,-0.00585,-165.8,165.8,-46.116667,46.116667,-1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9606.ENSP00000356694,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.026017,0.079951,-0.062079,-0.00585,-165.9,165.9,-47.916667,47.916667,2,0.0
9606.ENSP00000340610,0.0,0.000000,0.0,0.0,0.147059,0.147541,0.055556,0.000000,0.000000,0.0,...,0.026017,0.079951,-0.062079,-0.00585,-165.9,165.9,-47.916667,47.916667,-1,0.0
9606.ENSP00000370867,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.026017,0.079951,-0.062079,-0.00585,-165.9,165.9,-47.916667,47.916667,2,0.0


In [80]:
# data.to_csv('../Data/Datasets/'+phenotype+'/train_node_attribute_LDLR_with_target_feature_no_diffuse_target_6.csv')